# 73 — Multi-Task LGBM: pEC50 + Emax + Counter-pEC50

Trains three separate LightGBM regressors sharing the same features. The auxiliary Emax and pEC50_null models provide soft signal; final predictions blend primary model + auxiliary residual correction.

In [1]:
import os, sys, warnings
os.environ["PYTHONIOENCODING"] = "utf-8"
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy import stats
from pathlib import Path

from pxr.data import load_train, load_test
from pxr.featurize import combined, impute
from pxr.eval import rae, scaffold_kfold_indices
from pxr.chem import bemis_murcko, standardize_smiles
from pxr.paths import DATA_PROCESSED, DATA_EXTERNAL, SUBMISSIONS

SEED = 42
N_FOLDS = 5
LGBM_PARAMS = dict(
    n_estimators=1000, num_leaves=64, learning_rate=0.05,
    min_child_samples=10, subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=0.1, random_state=SEED,
    verbose=-1, n_jobs=4,
)


In [2]:
def full_metrics(y_true, y_pred, cliff_pairs_df=None, label=""):
    """RAE, MAE, R², Pearson, Spearman, Kendall, Cliff_accuracy."""
    yt = np.asarray(y_true, dtype=float)
    yp = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(yt) & np.isfinite(yp)
    yt, yp = yt[mask], yp[mask]

    mae_v  = float(np.mean(np.abs(yt - yp)))
    rae_v  = mae_v / float(np.mean(np.abs(yt - yt.mean()))) if yt.std() > 0 else float("nan")
    ss_res = float(np.sum((yt - yp) ** 2))
    ss_tot = float(np.sum((yt - yt.mean()) ** 2))
    r2_v   = 1.0 - ss_res / ss_tot if ss_tot > 0 else float("nan")
    pr_v, _ = stats.pearsonr(yt, yp)
    sp_v, _ = stats.spearmanr(yt, yp)
    kt_v, _ = stats.kendalltau(yt, yp)

    m = dict(RAE=rae_v, MAE=mae_v, R2=r2_v,
             Pearson=pr_v, Spearman=sp_v, Kendall=kt_v)

    if cliff_pairs_df is not None and len(cliff_pairs_df) > 0:
        correct = total = 0
        for _, row in cliff_pairs_df.iterrows():
            ia, ii = int(row.get("idx_active", -1)), int(row.get("idx_inactive", -1))
            if 0 <= ia < len(yp) and 0 <= ii < len(yp):
                correct += int(yp[ia] > yp[ii])
                total   += 1
        m["Cliff_acc"] = correct / total if total else float("nan")

    if label:
        cliff_str = f"  Cliff_acc={m.get('Cliff_acc', float('nan')):.3f}" if "Cliff_acc" in m else ""
        print(f"  [{label}] RAE={rae_v:.4f}  MAE={mae_v:.4f}  R²={r2_v:.4f}  "
              f"Pearson={pr_v:.4f}  Spearman={sp_v:.4f}  Kendall={kt_v:.4f}{cliff_str}")
    return m


In [3]:
tr = load_train()
te = load_test()
print(f"CRC train: {len(tr):,}  |  Test: {len(te):,}")

X_tr = impute(combined(tr["smiles"].tolist()))
X_te = impute(combined(te["smiles"].tolist()))
y_tr = tr["pec50"].values.astype(np.float64)
scaffolds = tr["smiles"].map(bemis_murcko).tolist()
splits = scaffold_kfold_indices(scaffolds, n_splits=N_FOLDS, seed=SEED)
active_mask = y_tr >= 5.5
print(f"X_tr: {X_tr.shape}  actives: {active_mask.sum()}")

cliff_pairs = (pd.read_parquet(DATA_PROCESSED / "cliff_pairs.parquet")
               if (DATA_PROCESSED / "cliff_pairs.parquet").exists()
               else pd.DataFrame())
print(f"Cliff pairs available: {len(cliff_pairs)}")


CRC train: 4,139  |  Test: 513


X_tr: (4139, 2265)  actives: 380
Cliff pairs available: 149


In [4]:
# Load auxiliary targets
emax = tr["emax"].values.astype(np.float32) if "emax" in tr.columns else None
emax_mask = np.isfinite(emax) if emax is not None else np.zeros(len(tr), dtype=bool)

from pxr.data import load_counter
from pxr.chem import to_inchikey
ctr = load_counter()
# Deduplicate counter by InChIKey to prevent row explosion on left-merge
ctr["ik"] = ctr["smiles"].map(to_inchikey)
ctr_dedup = ctr[["ik","pec50"]].dropna(subset=["ik"]).groupby("ik", as_index=False)["pec50"].mean()
tr["ik"] = tr["smiles"].map(to_inchikey)
tr_ctr = tr.merge(ctr_dedup.rename(columns={"pec50":"pec50_null"}), on="ik", how="left")
# Ensure no row explosion from the merge
assert len(tr_ctr) == len(tr), f"Merge expanded rows: {len(tr_ctr)} vs {len(tr)}"
pec50_null = tr_ctr["pec50_null"].values.astype(np.float32)
null_mask = np.isfinite(pec50_null)
print(f"Emax available: {emax_mask.sum()},  pEC50_null available: {null_mask.sum()}")

def run_auxiliary_cv(X, y, mask, splits, label):
    """Train an auxiliary LGBM regressor on compounds where target is available."""
    oof = np.full(len(y), np.nan)
    for fold, (tr_idx, va_idx) in enumerate(splits):
        tr_fold = tr_idx[mask[tr_idx]]
        va_fold = va_idx[mask[va_idx]]
        if len(tr_fold) < 20 or len(va_fold) < 5:
            continue
        m = lgb.train(LGBM_PARAMS,
                      lgb.Dataset(X[tr_fold], label=y[tr_fold]),
                      valid_sets=[lgb.Dataset(X[va_fold], label=y[va_fold])],
                      callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
        oof[va_idx] = m.predict(X[va_idx])
    valid = mask & np.isfinite(oof)
    if valid.sum() > 10:
        from pxr.eval import rae as _rae
        print(f"  {label} OOF RAE (where available): {_rae(y[valid], oof[valid]):.4f}")
    return oof

Emax available: 4139,  pEC50_null available: 2647


In [5]:
# Primary pEC50 CV
print("=== Primary: pEC50 ===")
oof_primary = np.full(len(y_tr), np.nan)
for fold, (tr_idx, va_idx) in enumerate(splits):
    m = lgb.train(LGBM_PARAMS, lgb.Dataset(X_tr[tr_idx], label=y_tr[tr_idx]),
                  valid_sets=[lgb.Dataset(X_tr[va_idx], label=y_tr[va_idx])],
                  callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
    oof_primary[va_idx] = m.predict(X_tr[va_idx])
    print(f"  fold {fold+1} RAE={rae(y_tr[va_idx], oof_primary[va_idx]):.4f}", flush=True)
m_primary = full_metrics(y_tr, oof_primary, cliff_pairs, "primary_pEC50")

# Auxiliary: Emax
oof_emax = None
if emax is not None and emax_mask.sum() > 100:
    print("\n=== Auxiliary: Emax ===")
    oof_emax = run_auxiliary_cv(X_tr, emax, emax_mask, splits, "Emax")

# Auxiliary: pEC50_null
oof_null = None
if null_mask.sum() > 100:
    print("\n=== Auxiliary: pEC50_null ===")
    oof_null = run_auxiliary_cv(X_tr, pec50_null, null_mask, splits, "pEC50_null")


=== Primary: pEC50 ===


  fold 1 RAE=0.4982


  fold 2 RAE=0.5759


  fold 3 RAE=0.6021


  fold 4 RAE=0.5665


  fold 5 RAE=0.6033


  [primary_pEC50] RAE=0.5643  MAE=0.5134  R²=0.5991  Pearson=0.7740  Spearman=0.7268  Kendall=0.5345  Cliff_acc=nan

=== Auxiliary: Emax ===


  Emax OOF RAE (where available): 0.8027

=== Auxiliary: pEC50_null ===


  pEC50_null OOF RAE (where available): 0.9471


In [6]:
# Blend: use auxiliary predictions as correction signal
# selectivity = pEC50 - pEC50_null; high selectivity = true PXR agonist
oof_blended = oof_primary.copy()
if oof_null is not None:
    valid_null = np.isfinite(oof_null)
    if valid_null.sum() > 50:
        # Selectivity correction: compounds predicted non-selective get penalized
        selectivity_pred = oof_primary - oof_null
        # Soft correction: shift toward 0 for non-selective compounds
        correction = np.where(selectivity_pred < 0.5, selectivity_pred * 0.15, 0.0)
        oof_blended = oof_primary + correction
        print(f"Selectivity correction applied to {valid_null.sum()} compounds")

m_blended = full_metrics(y_tr, oof_blended, cliff_pairs, "blended_multitask")
m_active_b = full_metrics(y_tr[active_mask], oof_blended[active_mask], label="blended [active]")
results_df = pd.DataFrame([m_primary, m_blended], index=["primary","blended"])
print("\n" + results_df.round(4).to_string())
oof = oof_blended


Selectivity correction applied to 4139 compounds
  [blended_multitask] RAE=0.5646  MAE=0.5137  R²=0.5979  Pearson=0.7732  Spearman=0.7266  Kendall=0.5344  Cliff_acc=nan
  [blended [active]] RAE=3.7193  MAE=0.7799  R²=-9.6139  Pearson=0.0697  Spearman=0.1061  Kendall=0.0729

            RAE     MAE      R2  Pearson  Spearman  Kendall  Cliff_acc
primary  0.5643  0.5134  0.5991   0.7740    0.7268   0.5345        NaN
blended  0.5646  0.5137  0.5979   0.7732    0.7266   0.5344        NaN


In [7]:
# Final models
m_final_primary = lgb.train(LGBM_PARAMS, lgb.Dataset(X_tr, label=y_tr),
                            callbacks=[lgb.log_evaluation(-1)])
te_primary = m_final_primary.predict(X_te)
# Emax auxiliary on test (if available)
te_preds = np.clip(te_primary, y_tr.min()-0.5, y_tr.max()+0.5)
np.save(DATA_PROCESSED / "oof_multitask_lgbm_heads.npy", oof)
np.save(DATA_PROCESSED / "te_oof_multitask_lgbm_heads.npy", te_preds)
sub = pd.DataFrame({"Molecule Name": te["name"].values, "pEC50": te_preds})
assert len(sub) == 513 and sub["pEC50"].notna().all()
out = SUBMISSIONS / "73_lgbm_multitask_heads.csv"
sub.to_csv(out, index=False)
print(f"Saved {out}")
print(f"Test preds  min={te_preds.min():.2f}  median={np.median(te_preds):.2f}  max={te_preds.max():.2f}")


Saved D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\submissions\73_lgbm_multitask_heads.csv
Test preds  min=2.35  median=4.95  max=6.00
